### 1.Local environment

In [20]:
#!/usr/bin/env python3
"""
diagnose_imputation_train_test_refined.py
"""

from __future__ import annotations

import json
from dataclasses import dataclass
from pathlib import Path
from typing import Optional, Sequence

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


matplotlib.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "Times", "DejaVu Serif"],
    "font.size": 10,
    "axes.labelsize": 10,
    "axes.titlesize": 11,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "legend.fontsize": 8,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.edgecolor": "#333333",
    "axes.linewidth": 0.8,
    "axes.grid": True,
    "grid.color": "#D9D9D9",
    "grid.linestyle": "--",
    "grid.linewidth": 0.5,
    "grid.alpha": 0.6,
    "savefig.facecolor": "white",
    "savefig.bbox": "tight",
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "axes.unicode_minus": False,
})

COLORS = {
    "observed": "#1f77b4",
    "gap": "#bdbdbd",
    "train": "#1f77b4",
    "imputed": "#ff7f0e",
    "test": "#6a3d9a",
    "gt": "#4d4d4d",
    "highlight_gap": "#d9d9d9",
}

SHOW_PLOTS = False
SAVE_PNG = True
SAVE_PDF = True
PNG_DPI = 300

TRAIN_PLOT_SCALE = "original"
N_TEST_FEATURES = 3

# DATASET = "python"
# DATASET = "golang"
DATASET = "amf"
# DATASET = "rabbitmq"

BASE_DIR = Path("./work/EUR")
PREPARED_DIR = BASE_DIR / f"prepared_{DATASET}"
GENERATED_DIR = BASE_DIR / f"generated_{DATASET}"
DIAG_DIR = GENERATED_DIR / "diagnostic_plots"

EVALUATED_FILE = "wavestitchPlus_full_imputed.csv"

# ── Helpers ──────────────────────────────────

def save_figure(fig: plt.Figure, path_base: Path) -> None:
    if SAVE_PNG:
        png_path = path_base.with_suffix(".png")
        fig.savefig(png_path, dpi=PNG_DPI)
        print(f"[SAVED] {png_path}")
    if SAVE_PDF:
        pdf_path = path_base.with_suffix(".pdf")
        fig.savefig(pdf_path)
        print(f"[SAVED] {pdf_path}")


def standardize(x: np.ndarray, mean: float, std: float) -> np.ndarray:
    return (x - mean) / (std + 1e-12)


def standardize_and_clip(x: np.ndarray, mean: float, std: float, clip: float = 3.0) -> np.ndarray:
    return np.clip(standardize(x, mean, std), -clip, clip)


def choose_center_from_variation(values: np.ndarray, obs_mask: np.ndarray) -> int:
    valid_idx = np.where(obs_mask & ~np.isnan(values))[0]
    if len(valid_idx) < 2:
        return len(values) // 2
    diff = np.abs(np.diff(values[valid_idx]))
    if len(diff) == 0:
        return len(values) // 2
    max_diff_idx = int(np.argmax(diff))
    return int(valid_idx[min(max_diff_idx + 1, len(valid_idx) - 1)])


def ensure_exists(path: Path, kind: str = "file") -> None:
    if not path.exists():
        raise FileNotFoundError(f"Missing {kind}: {path}")


def finite_minmax(*arrays: np.ndarray) -> tuple[float, float]:
    vals = []
    for arr in arrays:
        arr = np.asarray(arr)
        finite = arr[np.isfinite(arr)]
        if finite.size > 0:
            vals.append(finite)
    if not vals:
        return 0.0, 1.0
    cat = np.concatenate(vals)
    return float(np.min(cat)), float(np.max(cat))


def padded_lims(*arrays: np.ndarray, frac: float = 0.05) -> tuple[float, float]:
    y_min, y_max = finite_minmax(*arrays)
    span = y_max - y_min
    if span < 1e-12:
        pad = 0.5 if abs(y_min) < 1e-12 else 0.05 * abs(y_min)
    else:
        pad = frac * span
    return y_min - pad, y_max + pad


def feature_available(feature_name: str, cols: Sequence[str], *, label: str) -> bool:
    if feature_name not in cols:
        print(f"[SKIP] {feature_name} not in {label}")
        return False
    return True


def axvspan_batch(ax: plt.Axes, x: np.ndarray, mask: np.ndarray, **kwargs) -> None:
    if not mask.any():
        return
    padded = np.concatenate([[False], mask.astype(bool), [False]])
    starts = np.where(~padded[:-1] & padded[1:])[0]
    ends = np.where(padded[:-1] & ~padded[1:])[0]
    for s, e in zip(starts, ends):
        ax.axvspan(x[s] - 0.5, x[e - 1] + 0.5, linewidth=0, **kwargs)


# ── PreparedFeature ───────────────────────────

@dataclass
class PreparedFeature:
    feature_name: str
    col_idx: int
    target_col_idx: int
    orig_col: np.ndarray        # original scale (NaN at gaps)
    obs_col_mask: np.ndarray    # True where observed
    miss_col_mask: np.ndarray   # True where gap
    imp_col: np.ndarray         # already in original scale (from denorm file)
    col_mean: float             # kept for normalized-scale distribution plot only
    col_std: float

    def get_plot_arrays(self, scale: str, clip_for_original_norm: bool = True) -> tuple[np.ndarray, np.ndarray, str, str]:
        scale = scale.lower()
        if scale == "original":
            orig_plot = self.orig_col.copy()
            imp_plot  = self.imp_col.copy()
            ylabel    = self.feature_name
            scale_tag = "original"
        elif scale == "normalized":
            if clip_for_original_norm:
                orig_plot = standardize_and_clip(self.orig_col, self.col_mean, self.col_std, clip=3.0)
            else:
                orig_plot = standardize(self.orig_col, self.col_mean, self.col_std)
            imp_plot  = standardize(self.imp_col, self.col_mean, self.col_std)
            ylabel    = f"{self.feature_name} (normalized)"
            scale_tag = "normalized"
        else:
            raise ValueError(f"Unsupported scale: {scale}")
        return orig_plot, imp_plot, ylabel, scale_tag


# ── Load metadata and tables ─────────────────

meta_path = PREPARED_DIR / "meta.json"
ensure_exists(meta_path)

with meta_path.open("r", encoding="utf-8") as f:
    meta = json.load(f)

time_col      = meta.get("time_col", "time")
target_cols   = meta.get("target_cols", [])
cond_cols     = meta.get("cond_cols", [])
all_model_cols = meta.get("all_model_cols", [])

print(f"{'='*70}")
print("DATA LOCATION DIAGNOSTIC")
print(f"{'='*70}")
print(f"Meta: {meta_path}")
print(f"Time column: {time_col}")
print(f"Target columns ({len(target_cols)}): {target_cols}")
print(f"Conditioning columns ({len(cond_cols)}): {cond_cols}")

train_path       = PREPARED_DIR / "train.csv"
test_gt_path     = PREPARED_DIR / "test_gt.csv"
test_input_path  = PREPARED_DIR / "test_input.csv"
train_imputed_denorm_path = PREPARED_DIR / "train_imputed_denorm.npy"
scaler_dir       = PREPARED_DIR / "scaler"
pred_path        = GENERATED_DIR / EVALUATED_FILE

ensure_exists(train_path)
ensure_exists(test_gt_path)
ensure_exists(test_input_path)
ensure_exists(train_imputed_denorm_path)   # now required

train_df   = pd.read_csv(train_path)
test_gt    = pd.read_csv(test_gt_path)
test_input = pd.read_csv(test_input_path)

train_imputed_denorm = np.load(train_imputed_denorm_path)
print(f"[INFO] Loaded train_imputed_denorm.npy: shape={train_imputed_denorm.shape}")

# scaler still needed for normalized-space distribution plot
scaler_mean = None
scaler_std  = None
if (scaler_dir / "mean.npy").exists() and (scaler_dir / "std.npy").exists():
    scaler_mean = np.load(scaler_dir / "mean.npy")
    scaler_std  = np.load(scaler_dir / "std.npy")
    print(f"[INFO] Loaded scaler mean/std: shape={scaler_mean.shape}")
else:
    print("[WARNING] scaler mean/std not found — normalized-scale plots will be skipped")

pred_wavestitch_plus: Optional[pd.DataFrame] = None
if pred_path.exists():
    pred_wavestitch_plus = pd.read_csv(pred_path)
    print(f"[INFO] Loaded WaveStitch+ prediction: shape={pred_wavestitch_plus.shape}")
else:
    print("[WARNING] WaveStitch+ prediction file not found")

print(f"\n[Row Counts]")
print(f"  train:                  {len(train_df)}")
print(f"  train_imputed_denorm:   {train_imputed_denorm.shape[0]}")
print(f"  test_gt:                {len(test_gt)}")
print(f"  test_input:             {len(test_input)}")
print(f"  WaveStitch+:            {len(pred_wavestitch_plus) if pred_wavestitch_plus is not None else 'N/A'}")

# model columns
if all_model_cols:
    model_cols = all_model_cols
else:
    model_cols = [c for c in train_df.columns if c != time_col]

# train data
train_data = train_df.drop(columns=[time_col], errors="ignore").copy()
_available = set(train_data.columns)
_missing_model_cols = [c for c in model_cols if c not in _available]
if _missing_model_cols:
    print(f"[WARNING] model_cols has columns absent from train_data: {_missing_model_cols}")
model_cols = [c for c in model_cols if c in _available]
train_data = train_data[model_cols]

valid_target_cols   = [c for c in target_cols if c in train_data.columns]
missing_target_cols = [c for c in target_cols if c not in train_data.columns]
if missing_target_cols:
    print(f"[WARNING] Missing target columns in train_data: {missing_target_cols}")

obs_mask = (~train_data[valid_target_cols].isna()).to_numpy().astype(np.float32)

print(f"\n[Observation Mask]")
print(f"  Shape: {obs_mask.shape}")
print(f"  Observation rate: {obs_mask.mean():.2%}")

original_full = pd.concat([train_df, test_gt], ignore_index=True)
print(f"  train+test_gt: {len(original_full)}")


# ── Alignment check ──────────────────────────

def check_and_align(imputed_df, original_full_df, train_df_, test_df_, name):
    if imputed_df is None:
        print(f"\n[{name} Alignment]  N/A")
        return None, None, "none"
    print(f"\n[{name} Alignment]")
    if len(imputed_df) == len(original_full_df):
        print("  ✓ Imputed is FULL data (train + test)")
        return imputed_df, imputed_df.iloc[len(train_df_):].reset_index(drop=True), "full"
    if len(imputed_df) == len(test_df_):
        print("  ✓ Imputed is TEST data only")
        return pd.concat([train_df_, imputed_df], ignore_index=True), imputed_df, "test_only"
    print(f"  ✗ Unknown format: {len(imputed_df)} rows")
    return None, None, "unknown"


wsp_full, wsp_test, wsp_mode = check_and_align(
    pred_wavestitch_plus, original_full, train_df, test_input, "WaveStitch+"
)

DIAG_DIR.mkdir(parents=True, exist_ok=True)


# ── Feature preparation ───────────────────────

def prepare_feature(feature_name: str) -> Optional[PreparedFeature]:
    if not feature_available(feature_name, valid_target_cols, label="target_cols"):
        return None
    if feature_name not in model_cols:
        print(f"[SKIP] {feature_name} not in model_cols")
        return None

    col_idx        = model_cols.index(feature_name)
    target_col_idx = valid_target_cols.index(feature_name)

    if train_imputed_denorm.shape[0] != len(train_data):
        print(f"[SKIP] {feature_name}: train_imputed_denorm row mismatch "
              f"({train_imputed_denorm.shape[0]} vs {len(train_data)})")
        return None
    if col_idx >= train_imputed_denorm.shape[1]:
        print(f"[SKIP] {feature_name}: col_idx {col_idx} out of bounds for train_imputed_denorm")
        return None

    # scaler optional — only needed for normalized-space plots
    col_mean = float(scaler_mean[target_col_idx]) if scaler_mean is not None else 0.0
    col_std  = float(scaler_std[target_col_idx])  if scaler_std  is not None else 1.0

    return PreparedFeature(
        feature_name   = feature_name,
        col_idx        = col_idx,
        target_col_idx = target_col_idx,
        orig_col       = train_data[feature_name].to_numpy().copy(),
        obs_col_mask   = obs_mask[:, target_col_idx].astype(bool),
        miss_col_mask  = ~obs_mask[:, target_col_idx].astype(bool),
        imp_col        = train_imputed_denorm[:, col_idx].copy(),   # already original scale
        col_mean       = col_mean,
        col_std        = col_std,
    )


# ── Plot 1: layout overview ───────────────────

def plot_timeline_bar() -> None:
    fig, ax = plt.subplots(figsize=(14, 4.8))
    datasets = [
        ("train.csv",               0,           len(train_df),                              COLORS["train"]),
        ("train_imputed_denorm.npy",0,           train_imputed_denorm.shape[0],              COLORS["imputed"]),
        ("test_gt.csv",             len(train_df), len(train_df) + len(test_gt),             COLORS["test"]),
        ("test_input.csv",          len(train_df), len(train_df) + len(test_input),          COLORS["gap"]),
    ]
    if pred_wavestitch_plus is not None:
        n = len(pred_wavestitch_plus)
        if n == len(train_df) + len(test_gt):
            datasets.append(("WaveStitch+ (full)", 0, n, COLORS["imputed"]))
        elif n == len(test_input):
            datasets.append(("WaveStitch+ (test)", len(train_df), len(train_df) + n, COLORS["imputed"]))
        else:
            datasets.append(("WaveStitch+ (?)", len(train_df), len(train_df) + n, COLORS["imputed"]))

    y_labels = []
    for y_pos, (name, start, end, color) in enumerate(datasets):
        width = end - start
        ax.barh(y_pos, width, left=start, height=0.6, color=color, alpha=0.85, edgecolor="black")
        if width > 0:
            ax.text(start + width / 2, y_pos, f"{width} rows",
                    ha="center", va="center", fontsize=9, fontweight="bold", color="white")
        y_labels.append(name)

    ax.axvline(x=len(train_df), color="black", linestyle="--", linewidth=1.5, label="Train/Test split")
    ax.set_yticks(range(len(y_labels)))
    ax.set_yticklabels(y_labels)
    ax.set_xlabel("Row index")
    ax.set_title(f"{DATASET} — Data layout overview", pad=8)
    ax.legend(frameon=True, loc="upper right")
    ax.grid(True, alpha=0.4, axis="x")
    plt.tight_layout()
    save_figure(fig, DIAG_DIR / "data_layout_overview")
    if SHOW_PLOTS: plt.show()
    plt.close(fig)


# ── Plot 2: train comparison (stacked) ───────

def plot_train_comparison(feature_name: str, window_size: int = 300,
                           center_idx: Optional[int] = None) -> None:
    prepared = prepare_feature(feature_name)
    if prepared is None:
        return

    orig_col_plot, imp_col_plot, ylabel, scale_tag = prepared.get_plot_arrays("original")

    if center_idx is None:
        center_idx = choose_center_from_variation(orig_col_plot, prepared.obs_col_mask)

    s = max(0, center_idx - window_size // 2)
    e = min(len(orig_col_plot), center_idx + window_size // 2)
    x          = np.arange(s, e)
    orig_slice = orig_col_plot[s:e]
    imp_slice  = imp_col_plot[s:e]
    obs_slice  = prepared.obs_col_mask[s:e]
    miss_slice = prepared.miss_col_mask[s:e]

    fig, axes = plt.subplots(3, 1, figsize=(16, 9.2), sharex=True,
                              gridspec_kw={"height_ratios": [0.22, 1, 1]})
    ax_mask, ax1, ax2 = axes

    axvspan_batch(ax_mask, x, obs_slice,  alpha=0.6, color=COLORS["observed"])
    axvspan_batch(ax_mask, x, ~obs_slice, alpha=0.6, color=COLORS["gap"])
    ax_mask.set_ylim(0, 1); ax_mask.set_yticks([])
    ax_mask.spines[["top","right","left","bottom"]].set_visible(False)
    ax_mask.grid(False)
    ax_mask.set_title(f"{feature_name} | Observed: {obs_slice.sum()}   Gaps: {miss_slice.sum()}",
                      loc="left", pad=4)

    axvspan_batch(ax1, x, miss_slice, alpha=0.10, color=COLORS["highlight_gap"])
    valid_obs = obs_slice & ~np.isnan(orig_slice)
    ax1.scatter(x[valid_obs], orig_slice[valid_obs], s=18, color=COLORS["observed"],
                alpha=0.85, label="Observed", zorder=4)
    ax1.set_ylabel(ylabel)
    ax1.set_title("Original train data", pad=6)
    ax1.legend(frameon=True, loc="upper right")

    axvspan_batch(ax2, x, miss_slice, alpha=0.10, color=COLORS["highlight_gap"])
    ax2.plot(x, imp_slice, color=COLORS["imputed"], linewidth=1.4, alpha=0.9,
             label="EM imputed (denorm)", zorder=2)
    ax2.scatter(x[obs_slice], imp_slice[obs_slice], s=14, color=COLORS["observed"],
                alpha=0.75, label="At observed", zorder=3)
    if miss_slice.any():
        ax2.scatter(x[miss_slice], imp_slice[miss_slice], s=24, color=COLORS["imputed"],
                    alpha=0.9, marker="s", edgecolors="black", linewidth=0.4,
                    label="At gap", zorder=4)
    ax2.set_ylabel(ylabel)
    ax2.set_xlabel("Index")
    ax2.set_title("EM-imputed train data (train_imputed_denorm.npy)", pad=6)
    ax2.legend(frameon=True, loc="upper right")

    y0, y1 = padded_lims(orig_slice, imp_slice)
    ax1.set_ylim(y0, y1); ax2.set_ylim(y0, y1)

    plt.tight_layout()
    save_figure(fig, DIAG_DIR / f"{feature_name}_train_comparison_original")
    if SHOW_PLOTS: plt.show()
    plt.close(fig)


# ── Plot 3: train overlay ─────────────────────

def plot_train_comparison_overlay(feature_name: str, window_size: int = 300,
                                   center_idx: Optional[int] = None) -> None:
    prepared = prepare_feature(feature_name)
    if prepared is None:
        return

    orig_col_plot, imp_col_plot, ylabel, _ = prepared.get_plot_arrays("original")

    if center_idx is None:
        center_idx = choose_center_from_variation(orig_col_plot, prepared.obs_col_mask)

    s = max(0, center_idx - window_size // 2)
    e = min(len(orig_col_plot), center_idx + window_size // 2)
    x          = np.arange(s, e)
    orig_slice = orig_col_plot[s:e]
    imp_slice  = imp_col_plot[s:e]
    obs_slice  = prepared.obs_col_mask[s:e]
    miss_slice = prepared.miss_col_mask[s:e]

    fig, ax = plt.subplots(figsize=(16, 5.8))
    axvspan_batch(ax, x, miss_slice, alpha=0.08, color=COLORS["highlight_gap"])
    ax.plot(x, imp_slice, color=COLORS["imputed"], linewidth=1.6, alpha=0.9,
            label="EM imputed (denorm)", zorder=2)
    valid_obs = obs_slice & ~np.isnan(orig_slice)
    ax.scatter(x[valid_obs], orig_slice[valid_obs], s=34, color=COLORS["observed"],
               alpha=0.9, marker="o", edgecolors="white", linewidth=0.4,
               label="Observed", zorder=4)
    if miss_slice.any():
        ax.scatter(x[miss_slice], imp_slice[miss_slice], s=28, color=COLORS["imputed"],
                   alpha=0.9, marker="s", edgecolors="black", linewidth=0.4,
                   label="Imputed at gap", zorder=5)

    y0, y1 = padded_lims(orig_slice, imp_slice)
    ax.set_ylim(y0, y1)
    ax.set_xlabel("Index"); ax.set_ylabel(ylabel)
    ax.set_title(f"{feature_name} — Overlay comparison (idx {s}-{e})", pad=8)
    ax.legend(frameon=True, loc="upper right")
    plt.tight_layout()
    save_figure(fig, DIAG_DIR / f"{feature_name}_overlay_orig")
    if SHOW_PLOTS: plt.show()
    plt.close(fig)


# ── Plot 4: consistency check ─────────────────

def check_consistency(feature_name: str) -> None:
    prepared = prepare_feature(feature_name)
    if prepared is None:
        return

    orig_col  = prepared.orig_col
    imp_col   = prepared.imp_col
    valid     = prepared.obs_col_mask & np.isfinite(orig_col) & np.isfinite(imp_col)

    if valid.sum() == 0:
        print(f"\n[{feature_name}] consistency check: no valid observed points")
        return

    orig_at_obs = orig_col[valid]
    imp_at_obs  = imp_col[valid]
    diff        = np.abs(orig_at_obs - imp_at_obs)

    print(f"\n[{feature_name}] consistency check (original scale):")
    print(f"  observed points:  {valid.sum()}")
    print(f"  original range:   [{orig_at_obs.min():.6f}, {orig_at_obs.max():.6f}]")
    print(f"  imputed range:    [{imp_at_obs.min():.6f}, {imp_at_obs.max():.6f}]")
    print(f"  diff: mean={diff.mean():.6f}, max={diff.max():.6f}")

    tol = 1e-3 * (orig_at_obs.max() - orig_at_obs.min() + 1e-12)
    if diff.max() > tol:
        pos = int(np.where(valid)[0][int(np.argmax(diff))])
        print(f"  ⚠ observed values seem modified (tol={tol:.6f})")
        print(f"    worst position: {pos}")
        print(f"    original: {orig_at_obs[int(np.argmax(diff))]:.6f}  imputed: {imp_at_obs[int(np.argmax(diff))]:.6f}")
    else:
        print("  ✓ observed values remain consistent")


# ── Plot 5: train distribution ────────────────

def plot_train_distribution(feature_name: str) -> None:
    prepared = prepare_feature(feature_name)
    if prepared is None:
        return

    orig_col = prepared.orig_col
    imp_col  = prepared.imp_col

    valid_obs = prepared.obs_col_mask & np.isfinite(orig_col) & np.isfinite(imp_col)
    valid_gap = prepared.miss_col_mask & np.isfinite(imp_col)

    observed_vals   = orig_col[valid_obs]
    imputed_at_obs  = imp_col[valid_obs]
    imputed_at_gap  = imp_col[valid_gap]

    if len(observed_vals) == 0 or len(imputed_at_gap) == 0:
        print(f"[SKIP] {feature_name}: insufficient values for distribution plot")
        return

    fig, axes = plt.subplots(1, 2, figsize=(13.5, 5.0))
    ax1, ax2  = axes

    all_vals = np.concatenate([observed_vals, imputed_at_gap])
    vmin, vmax = finite_minmax(all_vals)
    bins = np.linspace(vmin - 0.5, vmax + 0.5, 20) if np.isclose(vmin, vmax) else np.linspace(vmin, vmax, 40)

    ax1.hist(observed_vals,  bins=bins, alpha=0.55, color=COLORS["observed"], density=True,
             label=f"Observed (n={len(observed_vals)})")
    ax1.hist(imputed_at_gap, bins=bins, alpha=0.55, color=COLORS["imputed"],  density=True,
             label=f"Imputed at gap (n={len(imputed_at_gap)})")
    ax1.set_xlabel("Value (original scale)")
    ax1.set_ylabel("Density")
    ax1.set_title("Distribution comparison", pad=6)
    ax1.legend(frameon=True)

    if len(observed_vals) > 1:
        corr = np.corrcoef(observed_vals, imputed_at_obs)[0, 1]
        mae  = np.mean(np.abs(observed_vals - imputed_at_obs))
    else:
        corr = mae = np.nan

    x0, x1 = padded_lims(observed_vals, imputed_at_obs, frac=0.03)
    lims = [min(x0, x1), max(x0, x1)]
    ax2.scatter(observed_vals, imputed_at_obs, s=8, alpha=0.35, color=COLORS["observed"])
    ax2.plot(lims, lims, linestyle="--", color="black", linewidth=1.0, label="y = x")
    ax2.set_xlim(lims); ax2.set_ylim(lims)
    ax2.set_xlabel("Original at observed")
    ax2.set_ylabel("Imputed at observed")
    ax2.set_title(f"Consistency | Corr={corr:.4f}, MAE={mae:.6f}", pad=6)
    ax2.legend(frameon=True)

    plt.tight_layout()
    save_figure(fig, DIAG_DIR / f"{feature_name}_train_distribution_original")
    if SHOW_PLOTS: plt.show()
    plt.close(fig)


# ── Plot 6: multiple gap-centered windows ────

def plot_train_multi_windows(feature_name: str, n_windows: int = 2,
                              window_size: int = 300) -> None:
    prepared = prepare_feature(feature_name)
    if prepared is None:
        return

    orig_plot, imp_plot, ylabel, _ = prepared.get_plot_arrays("original")
    missing_indices = np.where(prepared.miss_col_mask)[0]

    if len(missing_indices) < n_windows:
        print(f"[SKIP] {feature_name}: not enough gaps ({len(missing_indices)} < {n_windows})")
        return

    centers = [int(missing_indices[i]) for i in
               np.linspace(0, len(missing_indices) - 1, n_windows, dtype=int)]

    fig, axes = plt.subplots(n_windows, 1, figsize=(16, 2.7 * n_windows), squeeze=False)
    axes = axes.flatten()

    for ax_idx, center in enumerate(centers):
        ax = axes[ax_idx]
        s  = max(0, center - window_size // 2)
        e  = min(len(orig_plot), center + window_size // 2)
        x          = np.arange(s, e)
        orig_slice = orig_plot[s:e]
        imp_slice  = imp_plot[s:e]
        obs_slice  = prepared.obs_col_mask[s:e]
        miss_slice = prepared.miss_col_mask[s:e]

        axvspan_batch(ax, x, miss_slice, alpha=0.08, color=COLORS["highlight_gap"])
        ax.plot(x, imp_slice, color=COLORS["imputed"], alpha=0.9, linewidth=1.3,
                label="EM imputed (denorm)")
        valid_obs = obs_slice & np.isfinite(orig_slice)
        if valid_obs.any():
            ax.scatter(x[valid_obs], orig_slice[valid_obs], s=24, color=COLORS["observed"],
                       alpha=0.85, label="Observed", zorder=3)
        if miss_slice.any():
            ax.scatter(x[miss_slice], imp_slice[miss_slice], s=20, color=COLORS["imputed"],
                       alpha=0.9, marker="s", zorder=4)

        y0, y1 = padded_lims(orig_slice, imp_slice)
        ax.set_ylim(y0, y1)
        ax.set_ylabel(ylabel)
        ax.set_title(f"Window {ax_idx+1}: idx {s}-{e} | gaps={miss_slice.sum()}", fontsize=10, pad=5)
        ax.legend(frameon=True, loc="upper right")

    axes[-1].set_xlabel("Index")
    plt.tight_layout()
    save_figure(fig, DIAG_DIR / f"{feature_name}_train_multi_windows_original")
    if SHOW_PLOTS: plt.show()
    plt.close(fig)


# ── Plot 7: full data overview ────────────────

def plot_data_overview(feature_name: str) -> None:
    if not feature_available(feature_name, original_full.columns, label="original_full"):
        return

    fig, axes = plt.subplots(2, 1, figsize=(16, 8.5), sharex=True)
    ax1, ax2  = axes
    train_len = len(train_df)

    train_vals = train_df[feature_name].to_numpy()
    test_vals  = test_gt[feature_name].to_numpy()

    ax1.plot(range(train_len), train_vals, color=COLORS["train"], alpha=0.85, linewidth=1.0, label="Train")
    ax1.plot(range(train_len, len(original_full)), test_vals, color=COLORS["test"],
             alpha=0.85, linewidth=1.0, label="Test (GT)")
    ax1.axvline(x=train_len, color="black", linestyle="--", linewidth=1.2, label="Train/Test split")
    ax1.set_ylabel(feature_name)
    ax1.set_title(f"Original data | train missing={int(np.isnan(train_vals).sum())}, "
                  f"test missing={int(np.isnan(test_vals).sum())}", pad=6)
    ax1.legend(frameon=True, loc="upper right")

    if wsp_full is not None and feature_name in wsp_full.columns:
        wsp_vals = wsp_full[feature_name].to_numpy()
        train_label = {"full": "Train (WaveStitch+ output)",
                       "test_only": "Train (copied original)"}.get(wsp_mode, "Train")
        ax2.plot(range(train_len), wsp_vals[:train_len], color=COLORS["train"],
                 alpha=0.85, linewidth=1.0, label=train_label)
        ax2.plot(range(train_len, len(wsp_full)), wsp_vals[train_len:], color=COLORS["imputed"],
                 alpha=0.9, linewidth=1.0, label="Test (WaveStitch+)")
        ax2.axvline(x=train_len, color="black", linestyle="--", linewidth=1.2)
        ax2.set_title(f"WaveStitch+ | remaining missing={int(np.isnan(wsp_vals).sum())}", pad=6)
        ax2.legend(frameon=True, loc="upper right")
    else:
        ax2.set_title("WaveStitch+ | N/A", pad=6)

    ax2.set_ylabel(feature_name)
    ax2.set_xlabel("Index")
    plt.tight_layout()
    save_figure(fig, DIAG_DIR / f"{feature_name}_data_overview")
    if SHOW_PLOTS: plt.show()
    plt.close(fig)


# ── Plot 8: test comparison ───────────────────

def plot_test_comparison(feature_name: str) -> None:
    if not feature_available(feature_name, test_input.columns, label="test_input"):
        return

    fig, ax = plt.subplots(figsize=(14, 5.5))
    n_test = len(test_input)
    x      = np.arange(n_test)

    gt_vals = test_gt[feature_name].to_numpy()
    ax.plot(x, gt_vals, color=COLORS["gt"], alpha=0.7, linewidth=1.2,
            linestyle=":", label="Ground truth")

    input_vals          = test_input[feature_name].to_numpy()
    observed_mask_test  = np.isfinite(input_vals)
    ax.scatter(x[observed_mask_test], input_vals[observed_mask_test],
               s=12, color=COLORS["observed"], alpha=0.85, label="Observed", zorder=3)

    extra = np.array([])
    if wsp_test is not None and feature_name in wsp_test.columns:
        wsp_vals = wsp_test[feature_name].to_numpy()
        ax.plot(x, wsp_vals, color=COLORS["imputed"], alpha=0.9,
                linewidth=1.2, label="WaveStitch+")
        extra = wsp_vals

    y0, y1 = padded_lims(gt_vals, input_vals, extra)
    ax.set_ylim(y0, y1)
    ax.set_xlabel("Index (test portion)")
    ax.set_ylabel(feature_name)
    ax.set_title(f"Test comparison | observed={observed_mask_test.sum()}, "
                 f"masked/gap={(~observed_mask_test).sum()}", pad=8)
    ax.legend(frameon=True, loc="upper right")
    plt.tight_layout()
    save_figure(fig, DIAG_DIR / f"{feature_name}_test_comparison")
    if SHOW_PLOTS: plt.show()
    plt.close(fig)


# ── Run ───────────────────────────────────────

print(f"\n{'='*70}")
print("GENERATING VISUALIZATIONS")
print(f"{'='*70}")
print(f"Output directory: {DIAG_DIR}")

print("\n[1] Data layout overview")
plot_timeline_bar()

print("\n[2] Train data comparison (original vs EM-imputed denorm)")
for feature_name in valid_target_cols:
    print(f"\n--- {feature_name} ---")
    check_consistency(feature_name)
    plot_train_comparison(feature_name, window_size=300)
    plot_train_comparison_overlay(feature_name, window_size=300)
    plot_train_distribution(feature_name)
    plot_train_multi_windows(feature_name, n_windows=2, window_size=300)

print("\n[3] Test and full-data comparison")
for feature_name in valid_target_cols[:N_TEST_FEATURES]:
    print(f"\n--- {feature_name} ---")
    plot_data_overview(feature_name)
    plot_test_comparison(feature_name)

print(f"\n{'='*70}")
print(f"[DONE] All diagnostic plots saved to: {DIAG_DIR}")
print(f"{'='*70}")


DATA LOCATION DIAGNOSTIC
Meta: work/EUR/prepared_amf/meta.json
Time column: time
Target columns (14): ['cpu_limit', 'cpu_usage', 'lat100_ms', 'lat50_ms', 'lat75_ms', 'lat80_ms', 'lat90_ms', 'lat95_ms', 'lat98_ms', 'lat99_ms', 'mean_ms', 'n', 'ram_limit_mb', 'ram_usage_mb']
Conditioning columns (8): ['t_norm', 'sin_day', 'cos_day', 'sin_hour', 'cos_hour', 'is_gap', 'time_since_last_obs', 'time_to_next_obs']
[INFO] Loaded train_imputed_denorm.npy: shape=(1970, 22)
[INFO] Loaded scaler mean/std: shape=(14,)
[INFO] Loaded WaveStitch+ prediction: shape=(493, 23)

[Row Counts]
  train:                  1970
  train_imputed_denorm:   1970
  test_gt:                493
  test_input:             493
  WaveStitch+:            493

[Observation Mask]
  Shape: (1970, 14)
  Observation rate: 32.74%
  train+test_gt: 2463

[WaveStitch+ Alignment]
  ✓ Imputed is TEST data only

GENERATING VISUALIZATIONS
Output directory: work/EUR/generated_amf/diagnostic_plots

[1] Data layout overview
[SAVED] work/EU